# Revision figures and tables from committed results (A23)

Draws the revision's key figures and writes its tables from the committed `results/*.json` only: no data, no LFS,
no model fits. Rerunning `23` after any rerun of the notebooks it reads (then `99_tie_out_revision`) refreshes every
figure and table; nothing is typed by hand.

- Figures go to `figures/rev_*.pdf` (vector) and `figures/rev_*.png` (200 dpi). `12_figures`'s files are not touched.
- Tables go to `tables/rev_*.tex` as LaTeX `booktabs` `tabular` fragments (no float, no caption), for
  `\input{tables/...}` in the manuscript.
- `results/23_revision_figures.json` maps each figure and table to its source results, their code commits and
  platforms.

| File | Source |
|---|---|
| `rev_leakage_hierarchy` | `11` (address vs tree split) and `22` (report-mates seen vs held out) |
| `rev_report_dependence` | `22` (dose-response, by category) |
| `rev_label_effects` | `16` (false positives on the initial list; clean negatives) |
| `rev_mimicry` | `18` |
| `rev_shap_by_category` | `09` |
| `rev_models_10split.tex` | `17` |
| `rev_ensemble_appendix.tex`, `rev_ensemble_weight_stats.json` | `17`, `06` |
| `rev_report_dependence.tex` | `22` |
| `rev_label_robustness.tex`, `rev_operating_points.tex` | `16` |
| `rev_mimicry.tex` | `18` |

## Setup

In [1]:
import os, re, shutil, subprocess, tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)

SOURCES = ['03_xgboost_sybil', '04_lightgbm_sybil', '06_cross_ensemble_sybil', '09_shap_importance', '11_split_comparison',
           '16_label_robustness_initial_list', '17_models_10_splits', '18_mimicry_stress', '22_report_dependence']
pv = sp.provenance(SOURCES)
print(pv.to_string(index=False))
assert pv['dependencies_unchanged'].all(), pv
R = {n: sp.load_results(n) for n in SOURCES}

FIG_DIR, TAB_DIR = 'figures', 'tables'
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)
# style and colours as in 12_figures
C_XGB, C_LGBM, C_LR, C_ENS = '#1565C0', '#2E7D32', '#B71C1C', '#6A1B9A'
C_SHARED = '#E65100'
C_RF = '#455A64'
C_HELD, C_SEEN = C_XGB, C_SHARED          # A15b conditions: blue vs orange, plus marker/line style (colour-blind safe)
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
SMALL = 8.5                                 # smallest text used (pt); figures are drawn at print size

FIGURES, TABLES = {}, {}
def platform_of(n):
    p = R[n].get('platform') or R[n].get('inputs', {}).get('platform')
    return f"{p['machine']} / {p['system']}" if p else 'not recorded (machine of the committed 00-12 results)'
def source(names):
    return [dict(results=f'results/{n}.json', code_commit=R[n]['code_commit'], platform=platform_of(n)) for n in names]
def note_platforms(names):
    plats = {platform_of(n) for n in names}
    return None if len(plats) == 1 else 'mixes platforms: ' + '; '.join(f'{n}: {platform_of(n)}' for n in names)

def save(fig, name, names, **meta):
    for ext, kw in [('pdf', {}), ('png', dict(dpi=200))]:
        fig.savefig(os.path.join(FIG_DIR, f'{name}.{ext}'), bbox_inches='tight', **kw)
    plt.close(fig)
    FIGURES[name] = dict(files=[f'{FIG_DIR}/{name}.pdf', f'{FIG_DIR}/{name}.png'], sources=source(names),
                         platform_note=note_platforms(names), **meta)
    print('Saved', name)

fmt3 = lambda x: f'{x:.3f}'
pm = lambda m, s: f'{m:.3f} $\\pm$ {s:.3f}'
sgn = lambda x: '0.000' if round(x, 3) == 0 else f'{x:+.3f}'   # signed difference; no '-0.000'
cnt = lambda x: f'{int(round(x)):,}'
def pval(p):
    return '$<$0.001' if p < 0.001 else f'{p:.3f}'

Code commit: 3b1c175


                        notebook  commit  dependencies_unchanged
                03_xgboost_sybil 0d9eacc                    True
               04_lightgbm_sybil 0d9eacc                    True
         06_cross_ensemble_sybil 0d9eacc                    True
              09_shap_importance 0d9eacc                    True
             11_split_comparison 0d9eacc                    True
16_label_robustness_initial_list a03cf26                    True
             17_models_10_splits 8d24f2e                    True
               18_mimicry_stress 5ab0bb1                    True
            22_report_dependence 7da3628                    True


## F1. Leakage hierarchy: address vs tree split (A); report-mates seen vs held out (B)

In [2]:
cmp = R['11_split_comparison']['comparison']
r22 = R['22_report_dependence']
prev = r22['composition_summary']['k_a_prevalence_mean']
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.5, 3.4), gridspec_kw=dict(width_ratios=[1.25, 1]))
# (A)
models = [('XGBoost', 'XGBoost (tuned, 3-seed)', C_XGB), ('LightGBM', 'LightGBM (tuned, 3-seed)', C_LGBM)]
mets = [('f1', 'F1'), ('ap', 'AP'), ('auroc', 'AUROC')]
w, x0 = 0.38, 0
ticks, labels = [], []
for mi, (lab, key, col) in enumerate(models):
    for j, (m, mlab) in enumerate(mets):
        x = x0 + j
        a, t = cmp[key][m]['random'], cmp[key][m]['group']
        ax.bar(x - w / 2, a, w, color=col, alpha=0.35, hatch='///', edgecolor=col, lw=0)
        ax.bar(x + w / 2, t, w, color=col)
        ax.text(x, max(a, t) + 0.02, f'{t - a:+.3f}', ha='center', fontsize=SMALL)
        ticks.append(x); labels.append(mlab)
    ax.text(x0 + 1, -0.17, lab, ha='center', transform=ax.get_xaxis_transform(), fontsize=10, color=col, fontweight='bold')
    x0 += 3.6
ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=SMALL + 0.5)
ax.set_ylim(0, 1.3); ax.set_yticks(np.arange(0, 1.01, 0.2)); ax.set_ylabel('Test metric (seed-42 split)')
ax.legend(handles=[Patch(facecolor='0.6', alpha=0.35, hatch='///', label='address split (random)'),
                   Patch(facecolor='0.35', label='gas-provision-tree split')], fontsize=SMALL, loc='upper center', ncol=2, frameon=False)
ax.set_title('A  Address vs tree split', loc='left', fontsize=11)
# (B): held out in dark grey (blue is XGBoost in panel A), seen in orange; markers differ too
C_HELD_B = '#444444'
pf = pd.DataFrame(r22['per_fold'])
tests22 = pd.DataFrame(r22['tests'])
pos = {'auroc': (0, 1), 'f1': (2.4, 3.4)}
for m, (xh, xs) in pos.items():
    lg = pf[pf.model == 'LightGBM'].pivot_table(index=['repeat', 'k'], columns='condition', values=m)
    for _, row in lg.iterrows():
        bx.plot([xh, xs], [row['held_out'], row['seen']], color='0.7', lw=0.7, zorder=1)
    bx.plot([xh, xs], [lg['held_out'].mean(), lg['seen'].mean()], color='0.2', lw=1.5, zorder=2)
    bx.scatter([xh], [lg['held_out'].mean()], s=70, marker='o', color=C_HELD_B, zorder=3)
    bx.scatter([xs], [lg['seen'].mean()], s=70, marker='s', color=C_SEEN, zorder=3)
    xg = pf[pf.model == 'XGBoost'].pivot_table(index=['repeat', 'k'], columns='condition', values=m)
    bx.scatter([xh + 0.12], [xg['held_out'].mean()], s=45, marker='o', facecolors='none', edgecolors=C_HELD_B, zorder=3)
    bx.scatter([xs + 0.12], [xg['seen'].mean()], s=45, marker='s', facecolors='none', edgecolors=C_SEEN, zorder=3)
    t = tests22[(tests22.model == 'LightGBM') & (tests22.metric == m)].iloc[0]
    bx.text((xh + xs) / 2, 1.04, f"{t['diff_mean']:+.3f}\n{t['seen_higher_n']}/15", ha='center', va='bottom', fontsize=SMALL)
bx.set_xticks([0, 1, 2.4, 3.4]); bx.set_xticklabels(['held\nout', 'seen', 'held\nout', 'seen'], fontsize=SMALL + 0.5)
for m, (xh, xs) in pos.items():
    bx.text((xh + xs) / 2, -0.30, m.upper() if m == 'auroc' else 'F1', ha='center', transform=bx.get_xaxis_transform(), fontsize=10)
bx.set_ylim(0, 1.22); bx.set_yticks(np.arange(0, 1.01, 0.2)); bx.set_xlim(-0.4, 3.9)
bx.legend(handles=[Line2D([], [], marker='o', ls='', color='0.3', label='LightGBM (mean)'),
                   Line2D([], [], marker='o', ls='', markerfacecolor='none', markeredgecolor='0.3', label='XGBoost (mean)')],
          fontsize=SMALL, loc='lower right', frameon=False)
bx.set_title('B  Report-mates in training', loc='left', fontsize=11)
B_NOTE = (f'Panel B: test rows are {prev:.1%} Sybil (large components always in training); '
          'compare conditions, not absolute values with panel A.')
fig.text(0.5, -0.10, B_NOTE, ha='center', fontsize=SMALL)
fig.tight_layout()
save(fig, 'rev_leakage_hierarchy', ['11_split_comparison', '22_report_dependence'], caption_note=B_NOTE,
     panel_B='LightGBM per fold-repeat (15 grey lines), means as filled markers; XGBoost means hollow')

Saved rev_leakage_hierarchy


## F2. Report dependence: dose-response (A) and by category (B)

In [3]:
dr = pd.DataFrame(r22['dose_response']); dr = dr[dr.model == 'LightGBM'].set_index('bin').loc[['0', '1–9', '10–99', '≥100']]
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.5, 3.4), gridspec_kw=dict(width_ratios=[1.3, 1]))
x = np.arange(len(dr)); w = 0.36
ax.bar(x - w / 2, dr['recall_held'], w, color=C_HELD, hatch='///', edgecolor='white', lw=0, label='recall, held out')
ax.bar(x + w / 2, dr['recall_seen'], w, color=C_SEEN, label='recall, seen')
ax2 = ax.twinx()
ax2.errorbar(x, dr['score_gain'], yerr=dr['score_gain_sd'], fmt='D', color='0.15', ms=5, capsize=3, lw=1, label='mean score gain (± SD)')
ax2.axhline(0, color='0.6', lw=0.6, ls=':')
ax2.set_ylim(-0.6, 1.6); ax2.set_ylabel('Score gain per Sybil', fontsize=10); ax2.spines['right'].set_visible(True)
ax.set_xticks(x)
ax.set_xticklabels([f"{lab}\nn = {cnt(r['sybils'])}" for lab, (_, r) in zip(['0: no\nreport-mates', '1–9\n', '10–99\n', '≥100\n'], dr.iterrows())],
                   fontsize=SMALL)
ax.set_ylim(0, 1.35); ax.set_yticks(np.arange(0, 1.01, 0.2)); ax.set_ylabel('Recall at validation threshold')
ax.set_xlabel('Report-mates of the test Sybil in k_b', fontsize=10)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=SMALL, loc='upper left', frameon=False)
ax.set_title('A  Dose-response (LightGBM)', loc='left', fontsize=11)
bc = pd.DataFrame(r22['by_category']); bc = bc[bc.model == 'LightGBM'].set_index('category').loc[['IxL', 'IxE', 'IxI']]
x = np.arange(len(bc))
for i, (c, r) in enumerate(bc.iterrows()):
    bx.plot([i - 0.18, i + 0.18], [r['auroc_held_out'], r['auroc_seen']], color='0.4', lw=1.2, zorder=1)
    bx.scatter([i - 0.18], [r['auroc_held_out']], s=60, marker='o', color=C_HELD, zorder=2, label='held out' if i == 0 else None)
    bx.scatter([i + 0.18], [r['auroc_seen']], s=60, marker='s', color=C_SEEN, zorder=2, label='seen' if i == 0 else None)
    bx.text(i, 1.0, f"{r['auroc_diff_mean']:+.3f}\n± {r['auroc_diff_sd']:.3f}\n{int(r['auroc_seen_higher_n'])}/15",
            ha='center', va='bottom', fontsize=SMALL)
bx.set_xticks(x)
bx.set_xticklabels([f"{c}\nn = {r['test_sybils_per_fold']:.0f}" for c, r in bc.iterrows()], fontsize=SMALL)
bx.set_xlabel('n = test Sybils per fold', fontsize=SMALL)
bx.set_ylim(0.70, 1.12); bx.set_yticks(np.arange(0.70, 1.001, 0.05)); bx.set_xlim(-0.6, 2.6)
bx.set_ylabel('Test AUROC (mean of 15)')
bx.legend(fontsize=SMALL, loc='lower left', frameon=False)
bx.set_title('B  AUROC by category (LightGBM)', loc='left', fontsize=11)
fig.tight_layout()
save(fig, 'rev_report_dependence', ['22_report_dependence'],
     caption_note=f'Pooled over 15 fold-repeats; test rows are {prev:.1%} Sybil. Panel B: AUROC seen − held out, mean ± SD, and seen '
                  'higher out of 15. IxE and IxI have too few test Sybils per fold for reliable category-level estimates.')

Saved rev_report_dependence


## F3. Label effects: what the false positives are (A); clean negatives (B)

In [4]:
r16 = R['16_label_robustness_initial_list']
fp = pd.DataFrame(r16['fp_composition']).sort_values('threshold', ascending=False).reset_index(drop=True)
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.5, 3.4), gridspec_kw=dict(width_ratios=[1.35, 1]))
x = np.arange(len(fp))
ax.bar(x, fp['fp_not_on_list'], color=C_XGB, label='not on initial list')
ax.bar(x, fp['fp_on_initial_list'], bottom=fp['fp_not_on_list'], color=C_SHARED, hatch='///', edgecolor='white', lw=0, label='on initial list')
ax.set_xticks(x); ax.set_xticklabels([f"{t:.2f}" + ('\nval.' if n else '') for t, n in zip(fp['threshold'], fp['note'])], fontsize=SMALL)
ax.set_xlabel('Threshold', fontsize=10); ax.set_ylabel('Test false positives')
ax2 = ax.twinx(); ax2.spines['right'].set_visible(True)
ax2.plot(x, fp['precision'], 'o-', color='0.15', ms=4, lw=1, label='precision')
ax2.plot(x, fp['precision_if_list_counted_sybil'], 's--', color='0.15', ms=4, lw=1, markerfacecolor='white', label='precision, list counted as Sybil')
ax2.set_ylim(0.5, 1.0); ax2.set_ylabel('Precision', fontsize=10)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=SMALL - 0.5, loc='upper center', bbox_to_anchor=(0.5, -0.38), ncol=2, frameon=False)
ax.set_title('A  Test false positives (baseline)', loc='left', fontsize=11)
t22 = pd.DataFrame(r16['table_2x2'])
summ16 = r16['summary']
groups = [('full test', 'full'), ('test without initial-list rows', 'clean')]
mods = [('Baseline', 'base', C_XGB, '///'), ('Variant A (clean negatives)', 'A', C_SHARED, '')]
w = 0.36; xs = []
for gi, (m, mlab) in enumerate([('f1', 'F1'), ('ap', 'AP')]):
    for ti, (tset, tag) in enumerate(groups):
        xc = gi * 2.4 + ti
        for mi, (mod, key, col, hat) in enumerate(mods):
            v = t22[(t22.model == mod) & (t22.test_set == tset)][m].iloc[0]
            xx = xc + (mi - 0.5) * w
            bx.bar(xx, v, w, color=col, hatch=hat, edgecolor='white', lw=0)
            bx.errorbar(xx, summ16[f'{key}_{tag}_{m}_mean'], yerr=summ16[f'{key}_{tag}_{m}_std'], fmt='.', color='0.1', ms=4, capsize=2, lw=0.8)
        xs.append((xc, f"{mlab}\n{'full' if tag == 'full' else 'w/o list'}"))
bx.set_xticks([a for a, _ in xs]); bx.set_xticklabels([b for _, b in xs], fontsize=SMALL)
bx.set_ylim(0, 1.3); bx.set_yticks(np.arange(0, 1.01, 0.2)); bx.set_ylabel('Test metric')
bx.legend(handles=[Patch(facecolor=C_XGB, hatch='///', edgecolor='white', label='baseline'),
                   Patch(facecolor=C_SHARED, label='clean negatives'),
                   Line2D([], [], marker='.', ls='', color='0.1', label='10-split mean ± SD')], fontsize=SMALL - 0.5, loc='upper left', frameon=False)
bx.set_title('B  Clean negatives', loc='left', fontsize=11)
fig.tight_layout()
save(fig, 'rev_label_effects', ['16_label_robustness_initial_list'],
     caption_note='Panel B: bars are the seed-42 split (3 model seeds); points are the 10-split mean ± SD (one model seed).')

Saved rev_label_effects


## F4. Mimicry stress test

In [5]:
r18 = R['18_mimicry_stress']
st = pd.DataFrame(r18['results'])
cs = st[st.features == 'cheap + structural'].set_index(['attack', 'k'])['recall_mean']
co = st[st.features == 'cheap only'].set_index(['attack', 'k'])['recall_mean']
same_le10 = bool(np.allclose(cs[cs.index.get_level_values('k') <= 10], co.loc[cs[cs.index.get_level_values('k') <= 10].index], atol=0))
fig, ax = plt.subplots(figsize=(5.0, 3.4))
for order, col, mk in [('all features', C_XGB, 'o'), ('cheap only', C_SHARED, 's')]:
    for attack, ls in [('M1 marginal', '-'), ('M2 profile copy', '--')]:
        t = st[(st.features == order) & (st.attack == attack)].sort_values('k')
        ax.errorbar(t['k'], t['recall_mean'], yerr=t['recall_std'], color=col, ls=ls, marker=mk, ms=4, lw=1.2, capsize=2,
                    markerfacecolor=col if ls == '-' else 'white', label=f"{attack.split()[0]} {attack.split(' ', 1)[1]}, {order}")
ax.axhline(r18['baseline']['recall'], color='0.3', lw=1, ls=':', label=f"no attack ({r18['baseline']['recall']:.3f})")
ax.set_xscale('log'); ax.set_xticks(r18['ks']); ax.set_xticklabels(r18['ks'])
ax.set_xlabel('Top-k features replaced (by mean |SHAP|)'); ax.set_ylabel('Recall of test Sybils\n(fixed threshold)')
ax.set_ylim(-0.02, 0.8); ax.legend(fontsize=SMALL - 0.5, frameon=False, loc='center right', bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
note18 = ('"cheap + structural" equals "cheap only" for k ≤ 10' if same_le10 else
          '"cheap + structural" differs from "cheap only" for some k ≤ 10')
save(fig, 'rev_mimicry', ['18_mimicry_stress'], caption_note=note18 + '; mean ± SD over 5 repetitions; feature cost classes tentative.',
     cheap_structural_equals_cheap_k_le_10=same_le10)
print(note18)

Saved rev_mimicry
"cheap + structural" equals "cheap only" for k ≤ 10


## F5. SHAP by category (top 15 features, share of each column)

In [6]:
sh = pd.DataFrame(R['09_shap_importance']['mean_abs_shap'])
COLS = ['All', 'IxL', 'IxI', 'IxE']
share = sh.set_index('feature')[COLS] / sh[COLS].sum().to_numpy()
top = sh.sort_values('All', ascending=False).head(15)
H = share.loc[top['feature']]
FAM_TAG = {'LayerZero transactions': 'LZ', 'Ethereum transactions': 'ETH', 'Gas provider': 'GP', 'Gas provision tree': 'TREE',
           'Provision chain': 'CHAIN'}
fig, ax = plt.subplots(figsize=(5.0, 5.2))
im = ax.imshow(H.to_numpy(), cmap='cividis', aspect='auto')
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.iat[i, j]
        ax.text(j, i, f'{v:.1%}', ha='center', va='center', fontsize=SMALL, color='white' if v < H.to_numpy().max() * 0.55 else 'black')
ax.set_xticks(range(len(COLS))); ax.set_xticklabels(COLS)
ax.set_yticks(range(len(H))); ax.set_yticklabels([f"{f}  [{FAM_TAG[fam]}]" for f, fam in zip(top['feature'], top['family'])], fontsize=SMALL)
ax.spines[:].set_visible(False); ax.tick_params(length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02); cb.ax.tick_params(labelsize=SMALL)
cb.set_label('Share of column total |SHAP|', fontsize=SMALL + 0.5)
fig.tight_layout()
save(fig, 'rev_shap_by_category', ['09_shap_importance'],
     caption_note='Mean |SHAP| on the test partition, each column normalized to its total over all 62 features; family tags: '
                  + ', '.join(f'{v} = {k}' for k, v in FAM_TAG.items()) + '.')

Saved rev_shap_by_category


F6 (`rev_models_10split` dot plot) is not drawn: Table T1 carries the same numbers.

## Tables

In [7]:
def write_table(name, colspec, header, rows, comment, names):
    """booktabs tabular fragment; rows are lists of cell strings, or a str for a full-width line ('\\midrule' or a section)."""
    lines = [f'% {comment}', f'% Generated by 23_revision_figures.ipynb at {sp.CODE_COMMIT} from ' + ', '.join(f'results/{n}.json' for n in names),
             f'\\begin{{tabular}}{{{colspec}}}', '\\toprule'] + ([' & '.join(header) + ' \\\\', '\\midrule'] if header else [])
    for r in rows:
        lines.append(r if isinstance(r, str) else ' & '.join(r) + ' \\\\')
    lines += ['\\bottomrule', '\\end{tabular}', '']
    text = '\n'.join(lines)
    with open(os.path.join(TAB_DIR, f'{name}.tex'), 'w') as f:
        f.write(text)
    TABLES[name] = dict(file=f'{TAB_DIR}/{name}.tex', sources=source(names), platform_note=note_platforms(names))
    print(f'--- {name}.tex'); print(text)
    return text

In [8]:
# T1: models over 10 splits (17)
r17 = R['17_models_10_splits']
ps = pd.DataFrame(r17['per_split'])
t17 = pd.DataFrame(r17['tests'])
ORDER = [('LightGBM', 'LightGBM'), ('XGBoost', 'XGBoost'), ('Random Forest', 'Random forest'),
         ('Logistic regression', 'Logistic regression')]   # the weighted average is in the appendix table (T6)
METS = ['precision', 'recall', 'f1', 'ap', 'auroc']
T1 = {}
rows = []
for key, lab in ORDER:
    g = ps[ps.model == key]
    assert len(g) == 10
    vals = {m: (g[m].mean(), g[m].std(ddof=1)) for m in METS}
    for m in METS:   # agrees with the summary stored by 17
        assert abs(vals[m][0] - r17['summary'][key][f'{m}_mean']) < 1e-12 and abs(vals[m][1] - r17['summary'][key][f'{m}_std']) < 1e-12
    p = '--' if key == 'LightGBM' else pval(t17[(t17.metric == 'f1') & (t17.model == key) & (t17.against == 'LightGBM')]['p_corrected_holm'].iloc[0])
    T1[lab] = [pm(*vals[m]) for m in METS] + [p]
    rows.append([lab] + T1[lab])
t1 = write_table('rev_models_10split', 'lrrrrrr', ['Model', 'Precision', 'Recall', 'F1', 'AP', 'AUROC', '$p$ (F1)'], rows,
                 'Mean $\\pm$ sample SD (ddof = 1) over 10 stratified group splits, one model seed each; p: corrected resampled t-test vs LightGBM (Nadeau and Bengio 2003), Holm-adjusted over the five comparisons per metric stored by 17 (four models vs LightGBM and XGBoost vs the weighted average).',
                 ['17_models_10_splits'])

--- rev_models_10split.tex
% Mean $\pm$ sample SD (ddof = 1) over 10 stratified group splits, one model seed each; p: corrected resampled t-test vs LightGBM (Nadeau and Bengio 2003), Holm-adjusted over the five comparisons per metric stored by 17 (four models vs LightGBM and XGBoost vs the weighted average).
% Generated by 23_revision_figures.ipynb at 3b1c175 from results/17_models_10_splits.json
\begin{tabular}{lrrrrrr}
\toprule
Model & Precision & Recall & F1 & AP & AUROC & $p$ (F1) \\
\midrule
LightGBM & 0.717 $\pm$ 0.022 & 0.710 $\pm$ 0.022 & 0.713 $\pm$ 0.006 & 0.761 $\pm$ 0.006 & 0.968 $\pm$ 0.002 & -- \\
XGBoost & 0.718 $\pm$ 0.012 & 0.696 $\pm$ 0.019 & 0.707 $\pm$ 0.008 & 0.753 $\pm$ 0.008 & 0.964 $\pm$ 0.002 & 0.457 \\
Random forest & 0.722 $\pm$ 0.019 & 0.684 $\pm$ 0.021 & 0.702 $\pm$ 0.006 & 0.754 $\pm$ 0.007 & 0.969 $\pm$ 0.001 & 0.320 \\
Logistic regression & 0.146 $\pm$ 0.004 & 0.582 $\pm$ 0.039 & 0.234 $\pm$ 0.004 & 0.160 $\pm$ 0.009 & 0.832 $\pm$ 0.003 & $<$0.001 \\
\bo

In [9]:
# T6: weighted average (cross-ensemble) appendix table (17) and blend-weight statistics (17, 06)
T6 = {}
def tst(model, against, m):
    return t17[(t17.metric == m) & (t17.model == model) & (t17.against == against)].iloc[0]
EM = [('f1', 'F1'), ('ap', 'AP'), ('auroc', 'AUROC')]
S17 = r17['summary']
T6['LightGBM mean'] = [pm(S17['LightGBM'][f'{m}_mean'], S17['LightGBM'][f'{m}_std']) for m, _ in EM]
T6['Weighted average mean'] = [pm(S17['Ensemble'][f'{m}_mean'], S17['Ensemble'][f'{m}_std']) for m, _ in EM]
T6['Average $-$ LightGBM'] = [f"{tst('Ensemble', 'LightGBM', m)['mean_diff']:+.4f} $\\pm$ {tst('Ensemble', 'LightGBM', m)['sd_diff']:.4f}" for m, _ in EM]   # four decimals: the differences are below 0.0005
T6['\\quad corrected $p$ (Holm)'] = [pval(tst('Ensemble', 'LightGBM', m)['p_corrected_holm']) for m, _ in EM]
T6['\\quad splits with average higher'] = [f"{int(tst('Ensemble', 'LightGBM', m)['higher_n'])}/10" for m, _ in EM]
T6['XGBoost $-$ average'] = [f"{tst('XGBoost', 'Ensemble', m)['mean_diff']:+.4f} $\\pm$ {tst('XGBoost', 'Ensemble', m)['sd_diff']:.4f}" for m, _ in EM]
T6['\\quad corrected $p$ (Holm) '] = [pval(tst('XGBoost', 'Ensemble', m)['p_corrected_holm']) for m, _ in EM]
t6 = write_table('rev_ensemble_appendix', 'lrrr', ['', 'F1', 'AP', 'AUROC'], [[k] + v for k, v in T6.items()],
                 'Appendix (tab:ens). 10 stratified group splits, one model seed each; mean $\\pm$ SD; p: corrected resampled t-test, Holm-adjusted over the five comparisons per metric stored by 17.',
                 ['17_models_10_splits'])
wts = np.array([x['xgb_weight'] for x in r17['split_info']])
vf = {float(k): v for k, v in R['06_cross_ensemble_sybil']['val_f1_by_weight'].items()}
w06 = R['06_cross_ensemble_sybil']['xgb_weight']
low = [v for k, v in vf.items() if k <= 0.2 + 1e-12]
WEIGHT_STATS = dict(
    split_weights=dict(values=wts.tolist(), min=float(wts.min()), max=float(wts.max()), mean=float(wts.mean()),
                       source='results/17_models_10_splits.json -> split_info.xgb_weight', platform=platform_of('17_models_10_splits')),
    seed42_curve=dict(selected_weight=w06, val_f1_at_w0=vf[0.0], val_f1_at_selected=vf[float(w06)],
                      val_f1_min_w_le_0_2=min(low), val_f1_max_w_le_0_2=max(low),
                      source='results/06_cross_ensemble_sybil.json -> val_f1_by_weight (rounded to 4 decimals there)',
                      platform=platform_of('06_cross_ensemble_sybil')),
    generated_by=f'23_revision_figures.ipynb at {sp.CODE_COMMIT}')
import json
with open(os.path.join(TAB_DIR, 'rev_ensemble_weight_stats.json'), 'w') as f:
    json.dump(WEIGHT_STATS, f, indent=2)
TABLES['rev_ensemble_weight_stats'] = dict(file=f'{TAB_DIR}/rev_ensemble_weight_stats.json',
                                           sources=source(['17_models_10_splits', '06_cross_ensemble_sybil']),
                                           platform_note=note_platforms(['17_models_10_splits', '06_cross_ensemble_sybil']))
print(json.dumps(WEIGHT_STATS, indent=1))

# T2: report dependence (22)
summ22 = r22['summary']
T2 = {}
rows = []
for model in ['LightGBM', 'XGBoost']:
    for m, lab in [('f1', 'F1'), ('ap', 'AP'), ('auroc', 'AUROC')]:
        t = tests22[(tests22.model == model) & (tests22.metric == m)].iloc[0]
        cells = [pm(summ22[model][f'{m}_held_out_mean'], summ22[model][f'{m}_held_out_std']),
                 pm(summ22[model][f'{m}_seen_mean'], summ22[model][f'{m}_seen_std']),
                 f"{t['diff_mean']:+.3f} $\\pm$ {t['diff_sd']:.3f}", f"{int(t['seen_higher_n'])}/15", pval(t['p_corrected'])]
        assert abs(t['held_out_mean'] - summ22[model][f'{m}_held_out_mean']) < 1e-12
        T2[(model, lab)] = cells
        rows.append([model if lab == 'F1' else '', lab] + cells)
    if model == 'LightGBM':
        rows.append('\\midrule')
t2 = write_table('rev_report_dependence', 'llrrrrr', ['Model', 'Metric', 'Held out', 'Seen', 'Seen $-$ held out', 'Seen higher', '$p$'], rows,
                 f"Mean $\\pm$ SD over 15 fold-repeats (5 folds x 3 repeats); same test rows ({r22['composition_summary']['k_a_prevalence_mean']:.1%} Sybil) and training size in both conditions; p: corrected repeated k-fold t-test, df = 14.",
                 ['22_report_dependence'])

--- rev_ensemble_appendix.tex
% Appendix (tab:ens). 10 stratified group splits, one model seed each; mean $\pm$ SD; p: corrected resampled t-test, Holm-adjusted over the five comparisons per metric stored by 17.
% Generated by 23_revision_figures.ipynb at 3b1c175 from results/17_models_10_splits.json
\begin{tabular}{lrrr}
\toprule
 & F1 & AP & AUROC \\
\midrule
LightGBM mean & 0.713 $\pm$ 0.006 & 0.761 $\pm$ 0.006 & 0.968 $\pm$ 0.002 \\
Weighted average mean & 0.713 $\pm$ 0.008 & 0.761 $\pm$ 0.007 & 0.968 $\pm$ 0.002 \\
Average $-$ LightGBM & -0.0003 $\pm$ 0.0053 & +0.0003 $\pm$ 0.0020 & -0.0002 $\pm$ 0.0005 \\
\quad corrected $p$ (Holm) & 0.941 & 0.860 & 1.000 \\
\quad splits with average higher & 6/10 & 8/10 & 4/10 \\
XGBoost $-$ average & -0.0062 $\pm$ 0.0038 & -0.0081 $\pm$ 0.0028 & -0.0039 $\pm$ 0.0014 \\
\quad corrected $p$ (Holm)  & 0.320 & 0.028 & 0.030 \\
\bottomrule
\end{tabular}

{
 "split_weights": {
  "values": [
   0.6,
   0.1,
   0.2,
   0.52,
   0.14,
   0.24,
   0.76,


In [10]:
# T3: label robustness (16)
c = r16['counts']; v = fp[fp['note'] != ''].iloc[0]
vb = r16['variant_B']
rows = ['\\multicolumn{4}{l}{\\emph{LayerZero initial list among the interactors}} \\\\',
        ['Initial-list addresses', f"\\multicolumn{{3}}{{r}}{{{cnt(c['list_size'])}}}"],
        ['Overlap with the bounty-accepted list', f"\\multicolumn{{3}}{{r}}{{{cnt(c['overlap_with_fcfs_list'])}}}"],
        ['Interactors on the list (all labeled non-Sybil)', f"\\multicolumn{{3}}{{r}}{{{cnt(c['interactors_on_list'])}}}"],
        ['Share of the non-Sybil class', f"\\multicolumn{{3}}{{r}}{{{c['share_of_negatives']:.1%}}}".replace('%', '\\%')],
        '\\midrule',
        f"\\multicolumn{{4}}{{l}}{{\\emph{{Test false positives at the validation threshold ({v['threshold']:.3f})}}}} \\\\",
        ['False positives', f"\\multicolumn{{3}}{{r}}{{{cnt(v['fp'])}}}"],
        ['\\quad of which on the initial list', f"\\multicolumn{{3}}{{r}}{{{cnt(v['fp_on_initial_list'])} ({v['fp_share_on_list']:.1%})}}".replace('%', '\\%')],
        ['Precision / if the list counted as Sybil', f"\\multicolumn{{3}}{{r}}{{{fmt3(v['precision'])} / {fmt3(v['precision_if_list_counted_sybil'])}}}"],
        '\\midrule',
        ['\\emph{Model, test set}', '\\emph{F1}', '\\emph{AP}', '\\emph{AUROC}']]
for _, r in t22.iterrows():
    rows.append([f"{'Baseline' if r['model'] == 'Baseline' else 'Clean negatives'}, {'full test' if r['test_set'] == 'full test' else 'without initial-list rows'}",
                 fmt3(r['f1']), fmt3(r['ap']), fmt3(r['auroc'])])
rows += ['\\midrule', [f"Initial list relabeled Sybil (exploratory; prevalence {vb['prevalence_test']:.1%})".replace('%', '\\%'),
                        fmt3(vb['f1']), fmt3(vb['ap']), fmt3(vb['auroc'])]]
write_table('rev_label_robustness', 'lrrr', None, rows,
            'LightGBM, 3 model seeds, seed-42 tree split; thresholds from validation. Clean negatives: initial-list non-Sybils removed from training and validation.',
            ['16_label_robustness_initial_list'])

# T4: operating points (16)
rows = [[f"{r['threshold']:.3f}" + (' (val.)' if r['note'] else ''), cnt(r['flagged']), cnt(r['tp']), cnt(r['fp']),
         cnt(r['fp_on_initial_list']), fmt3(r['precision']), fmt3(r['precision_if_list_counted_sybil'])] for _, r in fp.iterrows()]
write_table('rev_operating_points', 'lrrrrrr', ['Threshold', 'Flagged', 'TP', 'FP', 'FP on list', 'Precision', 'Precision$^\\dagger$'], rows,
            'Test partition, baseline LightGBM (3 seeds). Precision-dagger: initial-list addresses counted as Sybil.',
            ['16_label_robustness_initial_list'])

# T5: mimicry (18)
KS = [1, 3, 5, 10]
rows = [['No attack'] + [fmt3(r18['baseline']['recall'])] + ['--'] * (len(KS) - 1)]
for order, olab in [('all features', 'all features'), ('cheap only', 'cheap only'), ('cheap + structural', 'cheap + structural')]:
    for attack, alab in [('M1 marginal', 'M1 marginal'), ('M2 profile copy', 'M2 profile copy')]:
        t = st[(st.features == order) & (st.attack == attack)].set_index('k')
        rows.append([f'{alab}, {olab}'] + [pm(t.loc[k, 'recall_mean'], t.loc[k, 'recall_std']) for k in KS])
write_table('rev_mimicry', 'l' + 'r' * len(KS), ['Attack, features'] + [f'$k={k}$' for k in KS], rows,
            'Recall of test Sybils at the fixed validation threshold after replacing the top-k features (by mean |SHAP|), mean $\\pm$ SD over 5 repetitions; no-attack recall in the k = 1 column.',
            ['18_mimicry_stress'])

--- rev_label_robustness.tex
% LightGBM, 3 model seeds, seed-42 tree split; thresholds from validation. Clean negatives: initial-list non-Sybils removed from training and validation.
% Generated by 23_revision_figures.ipynb at 3b1c175 from results/16_label_robustness_initial_list.json
\begin{tabular}{lrrr}
\toprule
\multicolumn{4}{l}{\emph{LayerZero initial list among the interactors}} \\
Initial-list addresses & \multicolumn{3}{r}{803,093} \\
Overlap with the bounty-accepted list & \multicolumn{3}{r}{0} \\
Interactors on the list (all labeled non-Sybil) & \multicolumn{3}{r}{34,545} \\
Share of the non-Sybil class & \multicolumn{3}{r}{8.3\%} \\
\midrule
\multicolumn{4}{l}{\emph{Test false positives at the validation threshold (0.600)}} \\
False positives & \multicolumn{3}{r}{1,461} \\
\quad of which on the initial list & \multicolumn{3}{r}{556 (38.1\%)} \\
Precision / if the list counted as Sybil & \multicolumn{3}{r}{0.727 / 0.831} \\
\midrule
\emph{Model, test set} & \emph{F1} & \emph

'% Recall of test Sybils at the fixed validation threshold after replacing the top-k features (by mean |SHAP|), mean $\\pm$ SD over 5 repetitions; no-attack recall in the k = 1 column.\n% Generated by 23_revision_figures.ipynb at 3b1c175 from results/18_mimicry_stress.json\n\\begin{tabular}{lrrrr}\n\\toprule\nAttack, features & $k=1$ & $k=3$ & $k=5$ & $k=10$ \\\\\n\\midrule\nNo attack & 0.711 & -- & -- & -- \\\\\nM1 marginal, all features & 0.410 $\\pm$ 0.006 & 0.074 $\\pm$ 0.002 & 0.012 $\\pm$ 0.001 & 0.000 $\\pm$ 0.000 \\\\\nM2 profile copy, all features & 0.410 $\\pm$ 0.006 & 0.114 $\\pm$ 0.001 & 0.036 $\\pm$ 0.003 & 0.005 $\\pm$ 0.001 \\\\\nM1 marginal, cheap only & 0.413 $\\pm$ 0.003 & 0.113 $\\pm$ 0.002 & 0.025 $\\pm$ 0.002 & 0.003 $\\pm$ 0.001 \\\\\nM2 profile copy, cheap only & 0.413 $\\pm$ 0.003 & 0.107 $\\pm$ 0.003 & 0.043 $\\pm$ 0.002 & 0.011 $\\pm$ 0.001 \\\\\nM1 marginal, cheap + structural & 0.413 $\\pm$ 0.003 & 0.113 $\\pm$ 0.002 & 0.025 $\\pm$ 0.002 & 0.003 $\\pm$ 0.001

## Checks

In [11]:
# every figure as .pdf and .png
for name, f in FIGURES.items():
    for p in f['files']:
        assert os.path.getsize(p) > 0, p
# numbers in T1 and T2 equal the JSON values after rounding: each expected cell appears in its row
def row_of(text, first):
    hits = [l for l in text.splitlines() if l.startswith(first + ' &')]
    assert len(hits) == 1, (first, hits); return hits[0]
for lab, cells in T1.items():
    line = row_of(t1, lab)
    assert [x.strip() for x in line.rstrip(' \\').split(' & ')[1:]] == cells, (lab, line)
assert not any(l.startswith('Cross-ensemble') for l in t1.splitlines())
for lab, cells in T6.items():
    line = row_of(t6, lab)
    assert [x.strip() for x in line.rstrip(' \\').split(' & ')[1:]] == cells, (lab, line)
lines2 = [l for l in t2.splitlines() if ' & ' in l and not l.startswith('Model')]
assert len(lines2) == 6
for (model, lab), cells in T2.items():
    hit = [l for l in lines2 if l.split(' & ')[1] == lab and l.split(' & ')[2] == cells[0]]
    assert len(hit) == 1 and [x.strip() for x in hit[0].rstrip(' \\').split(' & ')[2:]] == cells, (model, lab)
# structural check of every table: each row has the column count of its tabular (multicolumn spans counted)
for name, t in TABLES.items():
    if not t['file'].endswith('.tex'):
        continue
    text = open(t['file']).read()
    ncol = len(re.search(r'\\begin\{tabular\}\{([a-z]+)\}', text).group(1))
    for l in text.splitlines():
        if l.endswith('\\\\'):
            spans = sum(int(s) - 1 for s in re.findall(r'\\multicolumn\{(\d+)\}', l))
            assert l.count(' & ') + 1 + spans == ncol, (name, l)
# compile check
if shutil.which('pdflatex'):
    with tempfile.TemporaryDirectory() as d:
        body = '\n'.join(f'\\input{{{os.path.abspath(t["file"])}}}\n\\bigskip\n' for t in TABLES.values() if t['file'].endswith('.tex'))
        open(os.path.join(d, 'w.tex'), 'w').write('\\documentclass{article}\n\\usepackage{booktabs}\n\\begin{document}\n' + body + '\\end{document}\n')
        rc = subprocess.run(['pdflatex', '-interaction=nonstopmode', 'w.tex'], cwd=d, capture_output=True).returncode
        assert rc == 0, 'pdflatex failed'
    COMPILE = 'pdflatex: all tables compile'
else:
    COMPILE = 'WARNING: pdflatex not installed; compile check skipped (structural column check passed)'
print(COMPILE)
print('Figures:', sorted(FIGURES)); print('Tables:', sorted(TABLES))

pdflatex: all tables compile
Figures: ['rev_label_effects', 'rev_leakage_hierarchy', 'rev_mimicry', 'rev_report_dependence', 'rev_shap_by_category']
Tables: ['rev_ensemble_appendix', 'rev_ensemble_weight_stats', 'rev_label_robustness', 'rev_mimicry', 'rev_models_10split', 'rev_operating_points', 'rev_report_dependence']


## Save the results record

In [12]:
sp.save_results([], '23_revision_figures', extra=dict(
    figures=FIGURES, tables=TABLES,
    sources={k: v['sources'] for k, v in {**FIGURES, **TABLES}.items()},
    platform_notes={k: v['platform_note'] for k, v in {**FIGURES, **TABLES}.items() if v['platform_note']},
    compile_check=COMPILE, skipped=['F6 rev_models_10split (Table T1 carries the same numbers)']))

Saved results/23_revision_figures.json
